# NOTEBOOK: 01_bronze_ingest

- PURPOSE: 
  - Ingest the raw electronics_dataset.xlsx file and write two raw Delta tables — one for products, one for vendors.
  - Layer Bronze = raw data, exactly as received, no transformations. <br>

# 0. Imports

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import pandas as pd


In [0]:
# Load utils
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"

exec(open(f"{UTILS_DIR}/config.py").read())

# 0. Create Schema

In [0]:
# Create schemas with new naming convention
spark.sql("CREATE SCHEMA IF NOT EXISTS main.techmart_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.techmart_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.techmart_gold")

print("✅ Schemas created")
print("  main.techmart_bronze")
print("  main.techmart_silver")
print("  main.techmart_gold")

# 1. Catalog and schema names

In [0]:
# -----------------------------------------------------------------------------
# Configuration: all paths and table names in one place
# -----------------------------------------------------------------------------

# Ingestion metadata — added to every Bronze record for auditability
# Defined here because it is specific to this notebook's execution context
INGESTED_AT   = datetime.now().isoformat()
SOURCE_SYSTEM = "TechMart Vendor Feed"

print("✅ Config loaded")
print(f"Products file : {PRODUCTS_FILE}")
print(f"Vendors file  : {VENDORS_FILE}")

# 2. Reading tables

In [0]:
# STEP 1 — Create Volume in new schema location
spark.sql("""
    CREATE VOLUME IF NOT EXISTS main.techmart_bronze.raw_files
    COMMENT 'Raw source files for TechMart catalog pipeline'
""")

print("✅ Volume created at: /Volumes/main/techmart_bronze/raw_files/")
print("Now go to Catalog → main → techmart_bronze → raw_files and upload the files again")

In [0]:

df_products_raw = pd.read_excel(PRODUCTS_FILE, sheet_name=0)
df_vendors_raw  = pd.read_excel(VENDORS_FILE,  sheet_name=0)


In [0]:

print("=== Products raw ===")
df_products_raw.head()


In [0]:

print("\n=== Vendors raw ===")
df_vendors_raw.head()

# 3. Preparing and saving Product Table

In [0]:
# Rename columns to snake_case and cast everything to string
# Bronze = raw data, no transformations, all types kept as string
# Transformations happen in Silver

df_products_raw = df_products_raw.rename(columns={
    "ID"                  : "product_id",
    "Product Description" : "product_description_raw",
    "Weight"              : "weight_raw",
    "Unit Price"          : "unit_price_raw"
})

# Cast all to string — Bronze preserves raw values exactly as received
df_products_raw = df_products_raw.astype(str)

# Add ingestion metadata columns
df_products_raw["_ingested_at"]   = INGESTED_AT
df_products_raw["_source_system"] = SOURCE_SYSTEM
df_products_raw["_source_file"]   = "electronics_dataset_products.xlsx"

print("✅ Columns renamed and metadata added")
print("Vendors columns :", df_vendors_raw.columns.tolist())

In [0]:
df_products_raw.head()

In [0]:
# Convert to Spark DataFrames and write as Delta tables
# mergeSchema=true enables schema evolution — if new columns arrive in future
# feeds, the table will automatically expand to accommodate them

spark_products = spark.createDataFrame(df_products_raw)

# Write Products Bronze table
(spark_products.write
    .format("delta")
    .mode("overwrite")
    # .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PRODUCTS))

print(f"✅ Written: {BRONZE_PRODUCTS}")

In [0]:
# CELL 6 — Validate: confirm row counts and preview data
print("=== VALIDATION ===")
print(f"raw_products row count: {spark.table(BRONZE_PRODUCTS).count()}")

print("\n--- Products preview ---")
display(spark.table(BRONZE_PRODUCTS))

# 4. Preparing and saving Vendors Table

In [0]:
df_vendors_raw = df_vendors_raw.rename(columns={
    "Product ID" : "product_id",
    "Vendor"     : "vendor_name_raw"
})

df_vendors_raw  = df_vendors_raw.astype(str)

df_vendors_raw["_ingested_at"]    = INGESTED_AT
df_vendors_raw["_source_system"]  = SOURCE_SYSTEM
df_vendors_raw["_source_file"]    = "electronics_dataset_vendors.xlsx"


print("✅ Columns renamed and metadata added")
print("Vendors columns :", df_vendors_raw.columns.tolist())

In [0]:
df_vendors_raw.head()

In [0]:
# Convert to Spark DataFrames and write as Delta tables
# mergeSchema=true enables schema evolution — if new columns arrive in future
# feeds, the table will automatically expand to accommodate them

spark_vendors  = spark.createDataFrame(df_vendors_raw)

# Write Vendors Bronze table
(spark_vendors.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    # .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_VENDORS))

print(f"✅ Written: {BRONZE_VENDORS}")

In [0]:
# Validate: confirm row counts and preview data
print("=== VALIDATION ===")
print(f"raw_vendors  row count: {spark.table(BRONZE_VENDORS).count()}")

print("\n--- Vendors preview ---")
display(spark.table(BRONZE_VENDORS))